# Whisper API Test

Tests for the unified inference + training API

In [5]:
import requests
import os

BASE_URL = "http://127.0.0.1:8000"
AUDIO_PATH = "notebooks/example.wav"

## 1. Health Check

In [6]:
response = requests.get(f"{BASE_URL}/health")
print(response.json())

{'status': 'healthy', 'device': 'cuda', 'model_status': 'loaded', 'is_training': False}


## 2. List Models

In [7]:
response = requests.get(f"{BASE_URL}/models")
print(response.json())

{'base_models': [], 'fine_tuned_models': ['run_20260305_122238', 'run_20260305_122541', 'run_20260305_122844', 'run_20260305_123105'], 'active': 'run_20260305_123105'}


## 3. Get Active Model

In [8]:
response = requests.get(f"{BASE_URL}/models/active")
print(response.json())

{'active_model': 'run_20260305_123105'}


## 4. Transcription

In [9]:
with open(AUDIO_PATH, "rb") as audio_file:
    files = {"file": (os.path.basename(AUDIO_PATH), audio_file, "audio/wav")}
    response = requests.post(f"{BASE_URL}/transcribe", files=files)

print(response.json())

{'text': 'now request to more good evening one nine seven heavy rp one four seven seven tower good morning wind zero eight degrees one two knots go to one nine runway one four qero and track eight hotel two two'}


## 5. Start Training

In [10]:
train_config = {
    "model_id": "openai/whisper-tiny",
    "num_epochs": 1,
    "batch_size": 1,
    "learning_rate": 1e-5
}

response = requests.post(f"{BASE_URL}/train", json=train_config)
result = response.json()
print(result)

job_id = result["job_id"]

{'job_id': 'fa9bfb07-86bd-4045-a4ea-b61dc9808521', 'status': 'started', 'message': 'Training job started'}


## 6. Check Training Status

In [16]:
import time

while True:
    response = requests.get(f"{BASE_URL}/train/{job_id}")
    status = response.json()
    print(f"Status: {status['status']}")
    
    if status['status'] in ['completed', 'failed', 'cancelled']:
        break
    
    time.sleep(10)

Status: completed


## 7. Switch Model

In [13]:
# Get latest fine-tuned model
response = requests.get(f"{BASE_URL}/models")
models = response.json()
print("Available models:", models)

if models['fine_tuned_models']:
    latest_model = models['fine_tuned_models'][-1]
    print(f"\nSwitching to: {latest_model}")
    
    response = requests.post(
        f"{BASE_URL}/models/active",
        json={"model_id": latest_model}
    )
    print(response.json())

Available models: {'base_models': [], 'fine_tuned_models': ['run_20260305_122238', 'run_20260305_122541', 'run_20260305_122844', 'run_20260305_123105', 'run_20260305_123721'], 'active': 'run_20260305_123721'}

Switching to: run_20260305_123721
{'message': 'Model switched successfully', 'active_model': 'run_20260305_123721'}


## 8. Transcription with Fine-tuned Model

In [14]:
with open(AUDIO_PATH, "rb") as audio_file:
    files = {"file": (os.path.basename(AUDIO_PATH), audio_file, "audio/wav")}
    response = requests.post(f"{BASE_URL}/transcribe", files=files)

print(response.json())

{'text': 'now request to more good evening one nine seven heavy rp one four seven seven tower good morning wind zero eight degrees one two knots go to one nine runway one four qero and track eight hotel two two'}


## 9. Upload Dataset

In [15]:
with open(AUDIO_PATH, "rb") as f:
    response = requests.post(
        f"{BASE_URL}/train/dataset",
        files={"files": f}
    )
print(response.json())

{'message': 'Dataset uploaded', 'path': '/tmp/dataset_upload/upload_20260305_123854.parquet', 'samples': 1}
